# Етап 1.1 — джерела вакансій

Що з'ясовуємо:
- які параметри фільтрації реально працюють у RSS DOU і Djinni;
- які поля є у фідах і на сторінках вакансій;
- як дістати вакансії з Happy Monday, де немає ні API, ні RSS.

Усі запити йдуть з паузою ≥1 с на хост і кешуються в `samples/raw/` (не комітиться).

In [1]:
import sys, json, re
from pathlib import Path
import pandas as pd
import feedparser

sys.path.insert(0, str(Path.cwd()))
from recon_utils import *

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. DOU — RSS

Фід: `https://jobs.dou.ua/vacancies/feeds/?<фільтри>` — ті самі параметри, що й у URL сторінки пошуку.

In [2]:
DOU_FEED = "https://jobs.dou.ua/vacancies/feeds/"

def dou_feed(**params):
    q = "&".join(k if v is True else f"{k}={v}" for k, v in params.items())
    return feedparser.parse(fetch(f"{DOU_FEED}?{q}"))

def links(feed):
    return {e.link.split("?")[0] for e in feed.entries}

dou_tests = {
    "category=Python": dict(category="Python"),
    "+exp=1-3": dict(category="Python", exp="1-3"),
    "+remote": dict(category="Python", remote=True),
    "+city=Львів": dict(category="Python", city="Львів"),
    "search=fastapi": dict(search="fastapi"),
    "category=Front End": dict(category="Front End"),
}
base = links(dou_feed(category="Python"))
rows = []
for name, p in dou_tests.items():
    f = dou_feed(**p)
    l = links(f)
    rows.append({"filter": name, "items": len(f.entries), "same_as_base": round(len(l & base) / max(len(l), 1), 2),
                 "first": f.entries[0].title[:90] if f.entries else None})
pd.DataFrame(rows)

,filter,items,same_as_base,first
0,category=Python,25,1.00,Інженер машинного навчання ML Python до IT-підрозділу ЗСУ в 12-й ОЦСП
1,+exp=1-3,25,0.12,Інженер машинного навчання ML Python до IT-підрозділу ЗСУ в 12-й ОЦСП
2,+remote,25,0.72,Middle/Senior Python Backend Engineer (Real-Time Data &amp; AI) в Suntech In...
3,+city=Львів,25,0.36,"Senior Python Developer (ERP / Odoo) в Gonser, Львів"
4,search=fastapi,3,0.00,"Middle Python Developer (FastAPI) в Flo Lux, $2200–2300, віддалено"
5,category=Front End,25,0.04,Front-End розробник JavaScript (VueJS) до IT-підрозділу ЗСУ в 12-й ОЦСП


Заголовок фіду DOU містить компанію, зарплату і місто: `<Посада> в <Компанія>, $2200–2800, Київ, віддалено`. Розбираємо його:

In [3]:
dou = dou_feed(category="Python")
dou_df = pd.DataFrame([parse_dou_title(e.title) | {"link": e.link.split("?")[0], "published": e.published} for e in dou.entries])
print("Поля RSS-запису DOU:", list(dou.entries[0].keys()))
print("Вакансій із зарплатою в заголовку:", dou_df.salary_raw.notna().sum(), "з", len(dou_df))
dou_df[["title", "company", "salary_min", "salary_max", "remote", "cities"]].head(10)

Поля RSS-запису DOU: ['title', 'title_detail', 'links', 'link', 'summary', 'summary_detail', 'published', 'published_parsed', 'id', 'guidislink']
Вакансій із зарплатою в заголовку: 3 з 25


,title,company,salary_min,salary_max,remote,cities
0,Інженер машинного навчання ML Python до IT-підрозділу ЗСУ,12-й ОЦСП,NaN,NaN,False,[]
1,Middle/Senior Python Backend Engineer (Real-Time Data & AI),Suntech Innovation,NaN,NaN,True,[Київ]
2,Senior Full-Stack Engineer (Angular&Python),Altamira,NaN,NaN,True,"[Київ, Братислава (Словаччина)]"
3,Middle Full Stack Developer,Rayton,2200.0,2800.0,True,[]
4,Senior Python Developer (ERP / Odoo),Gonser,NaN,NaN,False,[Львів]
5,QA Automation Engineer (Python / API / Mobile) | Content Farm,WorkHold.AI,1300.0,1500.0,True,[]
6,R&D Infrastructure Engineer in Software Infrastructure Team (#1228),ZONE3000,NaN,NaN,True,[]
7,Engineering Manager (Python),AltexSoft,NaN,NaN,False,[]
8,Full-Stack Engineer (Python / React),Sigma Software,NaN,NaN,True,"[Київ, Львів, Вінниця, Івано-Франківськ, Тернопіль, Чернівці, Варшава (Польщ..."
9,Middle/Senior Python Engineer,Evolved Ideas,NaN,NaN,True,[]


Сторінка вакансії DOU: повний опис, місце, дата і **тип відгуку** (форма на DOU чи зовнішній сайт).

In [4]:
dou_pages = [parse_dou_page(fetch(u)) | {"url": u} for u in dou_df.link[:5]]
pd.DataFrame(dou_pages)[["title", "company", "place", "salary", "apply_type", "apply_href"]]

,title,company,place,salary,apply_type,apply_href
0,Інженер машинного навчання ML Python до IT-підрозділу ЗСУ,12-й ОЦСП,NaN,NaN,on_site,#apply
1,Middle/Senior Python Backend Engineer (Real-Time Data & AI),Suntech Innovation,"Київ, за кордоном, віддалено",NaN,external,https://dou.ua/goto/vacancy/?id=376945
2,Senior Full-Stack Engineer (Angular&Python),Altamira,"Київ, Братислава (Словаччина), віддалено",NaN,external,https://dou.ua/goto/vacancy/?id=376921
3,Middle Full Stack Developer,Rayton,віддалено,$2200–2800,on_site,#apply
4,Senior Python Developer (ERP / Odoo),Gonser,Львів,NaN,on_site,#apply


## 2. Djinni — RSS

Фід: `https://djinni.co/jobs/rss/?<фільтри>`. Перевіряємо, які параметри впливають на видачу.

In [5]:
DJ_FEED = "https://djinni.co/jobs/rss/"

def dj_feed(**params):
    q = "&".join(f"{k}={v}" for k, v in params.items())
    return feedparser.parse(fetch(f"{DJ_FEED}?{q}"))

dj_tests = {
    "primary_keyword=Python": dict(primary_keyword="Python"),
    "+exp_level=2y": dict(primary_keyword="Python", exp_level="2y"),
    "+salary=4000": dict(primary_keyword="Python", salary=4000),
    "+employment=remote": dict(primary_keyword="Python", employment="remote"),
    "+english_level=upper": dict(primary_keyword="Python", english_level="upper"),
    "+page=2": dict(primary_keyword="Python", page=2),
    "primary_keyword=React.js": dict(primary_keyword="React.js"),
    "primary_keyword=React (невалідне)": dict(primary_keyword="React"),
}
base = links(dj_feed(primary_keyword="Python"))
rows = []
for name, p in dj_tests.items():
    f = dj_feed(**p)
    l = links(f)
    rows.append({"filter": name, "items": len(f.entries), "same_as_base": round(len(l & base) / max(len(l), 1), 2),
                 "first": f.entries[0].title[:90] if f.entries else None})
pd.DataFrame(rows)

,filter,items,same_as_base,first
0,primary_keyword=Python,100,1.00,Python Developer
1,+exp_level=2y,15,0.80,Middle Python Developer
2,+salary=4000,61,0.84,Senior Python Backend Engineer
3,+employment=remote,100,0.88,Python Developer
4,+english_level=upper,78,0.87,Python Developer
5,+page=2,100,1.00,Python Developer
6,primary_keyword=React.js,12,0.00,"React Developer (trainee, junior, strong junior)"
7,primary_keyword=React (невалідне),100,0.02,Senior Affiliate Manager


Фід віддає максимум 100 записів. Якщо кількість зменшилась або `same_as_base < 1`, параметр працює. `same_as_base = 1.0` при тих самих 100 записах означає, що параметр **ігнорується** (так з `page`).

**Пастка:** невідоме значення `primary_keyword` (наприклад `React` замість `React.js`) **мовчки повертає всі вакансії**. Допустимі значення перелічені в тегах `<category>` самого RSS-каналу, їх використовуємо як словник.

In [6]:
cats = [t["term"] for t in dj_feed().feed.get("tags", [])]
print("Категорій у RSS-каналі Djinni:", len(cats))
print([c for c in cats if re.search(r"python|react|java|front|qa|data|ml|devops", c, re.I)])

Категорій у RSS-каналі Djinni: 170
['Data Analyst', 'Data Analytics', 'Data Engineer', 'Data Science', 'DevOps', 'Java', 'JavaScript', 'ML AI', 'Python', 'QA', 'QA Automation', 'QA Hardware / Embedded', 'React.js', 'React Native']


Деталі запису Djinni:

Запис RSS Djinni не містить компанії і зарплати, зате сторінка вакансії має **JSON-LD `JobPosting`** з компанією, досвідом у місяцях, типом роботи і терміном дії.

In [7]:
dj = dj_feed(primary_keyword="Python")
print("Поля RSS-запису Djinni:", list(dj.entries[0].keys()))
dj_pages = [parse_djinni_page(fetch(e.link)) | {"url": e.link} for e in dj.entries[:5]]
print("Ключі JSON-LD:", dj_pages[0]["jsonld_keys"])
pd.DataFrame(dj_pages)[["title", "company", "experience_months", "location_type", "salary", "valid_through", "direct_apply"]]

Поля RSS-запису Djinni: ['title', 'title_detail', 'links', 'link', 'summary', 'summary_detail', 'published', 'published_parsed', 'id', 'guidislink', 'tags']
Ключі JSON-LD: ['@context', '@type', 'applicantLocationRequirements', 'category', 'datePosted', 'description', 'directApply', 'employmentType', 'estimatedSalary', 'experienceRequirements', 'hiringOrganization', 'identifier', 'industry', 'jobLocationType', 'title', 'url', 'validThrough']


,title,company,experience_months,location_type,salary,valid_through,direct_apply
0,Python Developer,PLANEKS,60.0,TELECOMMUTE,"{'@type': 'MonetaryAmount', 'currency': 'USD', 'maxValue': 5000, 'minValue':...",2026-11-15T20:44:09.666586,True
1,Senior Python Backend Engineer,Verbit,84.0,TELECOMMUTE,"{'@type': 'MonetaryAmount', 'currency': 'USD', 'maxValue': 5000, 'minValue':...",2026-11-15T17:12:07.439667,True
2,Full-Stack (Python+React) Engineer,Temy,60.0,TELECOMMUTE,"{'@type': 'MonetaryAmount', 'currency': 'USD', 'maxValue': 5000, 'minValue':...",2026-11-15T17:10:10.686534,True
3,Staff Backend Engineer (Python) / LUN App,ЛУН,108.0,TELECOMMUTE,"{'@type': 'MonetaryAmount', 'currency': 'USD', 'maxValue': 5000, 'minValue':...",2026-11-15T16:30:18.186884,True
4,Middle Python Developer,Ajax Systems,24.0,NaN,None,2026-11-15T11:40:14.672649,True


In [8]:
print("Поля форми відгуку Djinni (видно без логіну):")
pd.DataFrame(dj_pages[0]["apply_form_fields"])

Поля форми відгуку Djinni (видно без логіну):


,tag,type,name,required
0,input,hidden,anon_apply,False
1,textarea,NaN,message,False
2,input,email,email,True
3,input,text,name,False
4,input,hidden,csrfmiddlewaretoken,False


## 3. Happy Monday — sitemap + HTML

API і RSS немає. Список усіх вакансій лежить у `https://happymonday.ua/job-sitemap.xml` (з `lastmod`), деталі — у HTML сторінки `/jobs/<id>`.

In [9]:
sm = fetch("https://happymonday.ua/job-sitemap.xml", ttl=3600)
hm_urls = re.findall(r"<loc>([^<]+)</loc>", sm)
hm_mod = re.findall(r"<lastmod>([^<]+)</lastmod>", sm)
print("Вакансій у sitemap:", len(hm_urls))
print("Найновіші lastmod:", hm_mod[:3])
hm_pages = [parse_hm_page(fetch(u)) | {"url": u} for u in hm_urls[:8]]
pd.DataFrame(hm_pages)[["title", "company", "meta", "partner_source", "apply_requires_login"]]

Вакансій у sitemap: 577
Найновіші lastmod: ['2026-10-17T11:32:22+00:00', '2026-10-17T10:52:52+00:00', '2026-10-17T10:52:51+00:00']


,title,company,meta,partner_source,apply_requires_login
0,Executive Operations Coordinator at MacPaw,MacPaw,"MacPaw Повна зайнятість Гібридний, В офісі Київ junior Upper-Intermediate (B...",False,True
1,Android Developer (AI Remodel),Kiss My Apps,Kiss My Apps Повна зайнятість Віддалений middle Англійська необов'язкова Рів...,True,True
2,Full Stack Developer (React/Next.js + Node.js),Kiss My Apps,Kiss My Apps Повна зайнятість Віддалений middle Англійська необов'язкова Рів...,True,True
3,HR Business Partner,Kiss My Apps,Kiss My Apps Повна зайнятість Віддалений middle Англійська необов'язкова Рів...,True,True
4,Enterprise Account Manager,Retell AI,Retell AI Повна зайнятість В офісі San Francisco middle Англійська необов'яз...,True,True
5,Procurement Manager,Retell AI,Retell AI Повна зайнятість В офісі San Francisco middle Англійська необов'яз...,True,True
6,Deployment Strategist,Retell AI,Retell AI Повна зайнятість В офісі San Francisco middle Англійська необов'яз...,True,True
7,Senior Affiliate Manager (Full Remote - Ukraine),EverAI,EverAI Повна зайнятість Віддалений senior Advanced Рівень англійської UA,True,True


## 4. Порівняння полів

Що можна отримати без логіну для кожного джерела.

In [10]:
def has(rows, key):
    vals = [r.get(key) for r in rows]
    filled = sum(v not in (None, "", [], {}) and not (isinstance(v, float) and v != v) for v in vals)
    return f"{filled}/{len(vals)}"

compare = pd.DataFrame({
    "DOU (RSS + сторінка)": {
        "назва": has(dou_pages, "title"), "компанія": has(dou_pages, "company"),
        "зарплата": has(dou_df.to_dict("records"), "salary_raw"), "місто / remote": has(dou_pages, "place"),
        "досвід": "з тексту", "повний опис": has(dou_pages, "description"), "тип відгуку": has(dou_pages, "apply_type"),
    },
    "Djinni (RSS + JSON-LD)": {
        "назва": has(dj_pages, "title"), "компанія": has(dj_pages, "company"),
        "зарплата": has(dj_pages, "salary"), "місто / remote": has(dj_pages, "location_type"),
        "досвід": has(dj_pages, "experience_months"), "повний опис": "RSS summary", "тип відгуку": has(dj_pages, "direct_apply"),
    },
    "Happy Monday (HTML)": {
        "назва": has(hm_pages, "title"), "компанія": has(hm_pages, "company"),
        "зарплата": "з тексту", "місто / remote": has(hm_pages, "meta"),
        "досвід": "meta (рівень)", "повний опис": has(hm_pages, "description"), "тип відгуку": "тільки після логіну",
    },
})
compare

,DOU (RSS + сторінка),Djinni (RSS + JSON-LD),Happy Monday (HTML)
назва,5/5,5/5,8/8
компанія,5/5,5/5,8/8
зарплата,3/25,4/5,з тексту
місто / remote,4/5,4/5,8/8
досвід,з тексту,5/5,meta (рівень)
повний опис,5/5,RSS summary,8/8
тип відгуку,5/5,5/5,тільки після логіну


In [11]:
save_sample("dou_sample.json", {"feed": dou_df.to_dict("records"), "pages": dou_pages})
save_sample("djinni_sample.json", {"pages": dj_pages})
save_sample("happymonday_sample.json", {"sitemap_count": len(hm_urls), "pages": hm_pages})
print("Збережено в samples/")

Збережено в samples/


Висновки зведено в `RECON.md`.